# Проверка генератора (EN + RU)

Свип curriculum, натуральные строки, скорость. Шрифты в `assets/fonts_ru` / `assets/fonts_en`.

In [ ]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
from src.synth import HandwrittenLineGenerator, make_generator

gen = HandwrittenLineGenerator.from_dirs(
    ru_text_dirs=[], en_text_dirs=[],
    ru_font_dirs=str(ROOT / 'assets' / 'fonts_ru'),
    en_font_dirs=str(ROOT / 'assets' / 'fonts_en'), p_ru=0.5)
print('fonts ru=%d en=%d | effects %s' % (gen.fonts.n('ru'), gen.fonts.n('en'), gen.effects.backend))

## Свип curriculum (строки = сложность t)

In [ ]:
rows, ncol = [0.0, 0.34, 0.67, 1.0], 5
fig, axes = plt.subplots(len(rows), ncol, figsize=(ncol * 3, len(rows) * 3))
draw = 0
for r, t in enumerate(rows):
    step = int(t * gen.cfg.warmup_steps)
    for c in range(ncol):
        img, text = gen.sample(make_generator(7, 0, draw), step); draw += 1
        axes[r][c].imshow(img); axes[r][c].axis('off')
        axes[r][c].set_title(f't={t:.2f}\n{text[:20]}', fontsize=8)
plt.tight_layout(); plt.show()

## Натуральные строки

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(11, 8))
for i, ax in enumerate(axes):
    img, text = gen.render_line(make_generator(21, 0, i), gen.cfg.warmup_steps)
    ax.imshow(img); ax.axis('off'); ax.set_title(text, fontsize=11)
plt.tight_layout(); plt.show()

## Пропускная способность

In [ ]:
n = 100; t0 = time.perf_counter()
for i in range(n):
    gen.sample(make_generator(3, 0, i), gen.cfg.warmup_steps)
dt = time.perf_counter() - t0
print(f'{n / dt:.0f} lines/s  ({1000 * dt / n:.2f} ms/line)')